# 4. Machine Learning Model & Evaluation

## Problem Classification
State whether the problem is regression, classification, or clustering.

Example:
This is a regression problem because the target variable is continuous traffic volume.

## Model Selection
Explain which models were tested and why.

Example:
Several models were tested, including Linear Regression, Random Forest, and Gradient Boosting. The final model was selected based on overall performance and the ability to generalize to unseen data.

## Evaluation & Validation
Explain metrics and validation methods used.

Example:
The dataset was split into training and testing sets. Model performance was evaluated using RMSE, MAE, and R² to ensure reliability and robustness.

In [105]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [106]:
df = pd.read_csv(r"D:\SIC_Zeina_801\Capstone_Project_2\Traffic-Volume\data\processed\traffic_volume_cleaned.csv")

In [107]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,day_of_week,month,hour_sin,hour_cos,month_sin,month_cos
0,Not Holiday,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,Tuesday,10,7.071068e-01,-0.707107,-0.866025,0.5
1,Not Holiday,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,Tuesday,10,5.000000e-01,-0.866025,-0.866025,0.5
2,Not Holiday,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,Tuesday,10,2.588190e-01,-0.965926,-0.866025,0.5
3,Not Holiday,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,Tuesday,10,1.224647e-16,-1.000000,-0.866025,0.5
4,Not Holiday,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,Tuesday,10,-2.588190e-01,-0.965926,-0.866025,0.5


In [108]:
# identify the features and target variable
X = df.drop('traffic_volume', axis=1)
y = df['traffic_volume']

In [109]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


In [110]:
print(df.columns.tolist())

['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'date_time', 'traffic_volume', 'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


In [111]:
df["date_time"] = pd.to_datetime(df["date_time"])

In [112]:
# Extract basic time features
df["hour"] = df["date_time"].dt.hour
df["month"] = df["date_time"].dt.month
df["day_of_week_num"] = df["date_time"].dt.dayofweek
df["year"] = df["date_time"].dt.year

In [113]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

In [114]:
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [115]:
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week_num"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week_num"] / 7)

In [116]:
print(df.isnull().sum())

holiday            0
temp               0
rain_1h            0
snow_1h            0
clouds_all         0
weather_main       0
date_time          0
traffic_volume     0
hour               0
day_of_week        0
month              0
hour_sin           0
hour_cos           0
month_sin          0
month_cos          0
day_of_week_num    0
year               0
day_sin            0
day_cos            0
dtype: int64


In [117]:
df["holiday"] = df["holiday"].apply(
    lambda x: "Holiday" if pd.notna(x) and x != "Not Holiday" else "Not Holiday"
)

In [118]:
df["holiday"] = df["holiday"].map({
    "Not Holiday": 0,
    "Holiday": 1
})

In [119]:
print(df["holiday"].value_counts(dropna=False))

holiday
0    39361
1     1203
Name: count, dtype: int64


In [120]:
df = df.sort_values("date_time").reset_index(drop=True)

In [121]:
features_to_drop = [
    "date_time",
    "hour",
    "month",
    "day_of_week",
    "day_of_week_num"
]

X = X.drop(columns=features_to_drop, errors="ignore")

In [122]:
print(X.columns.tolist())

['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


In [123]:
# train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [124]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [125]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 32451
Testing samples: 8113


### Define Categorical and Numerical Features

In [126]:
categorical_features = [
    "weather_main"
]

In [127]:
numerical_features = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "year",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
]

### Modeling

In [128]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor

In [129]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print(X_train.columns.tolist())

X_train: (32451, 10)
X_test: (8113, 10)
['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


In [130]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "weather_main"
]

numerical_features = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "year",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
]

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [131]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(
    steps=[
        ("preprocessor", linear_preprocessor),
        ("model", LinearRegression())
    ]
)

In [132]:
linear_model.fit(X_train, y_train)

ValueError: Some column names are not columns of the dataframe: {'day_sin', 'year', 'day_cos'}